# Running simulations through the cloud 

This notebook is a tutorial of the API used for submitting simulations to our servers.


In [1]:
import tidy3d as td
import tidy3d.web as web

# set logging level to ERROR because we'll only running simulations to demonstrate the web API, we dont care about the results.
td.config.logging_level = "ERROR"


## Setup

Let's set up a simple simulation to get started.

In [2]:
# whether to print output in web functions, note: if False (default) they will all run silently
verbose = True

# set up parameters of simulation
dl = 0.05
pml = td.PML()
sim_size = [4, 4, 4]
freq0 = 3e14
fwidth = 1e13
run_time = 1 / fwidth

# create structure
dielectric = td.Medium.from_nk(n=2, k=0, freq=freq0)
square = td.Structure(
    geometry=td.Box(center=[0, 0, 0], size=[1.5, 1.5, 1.5]), medium=dielectric
)

# create source
source = td.UniformCurrentSource(
    center=(-1.5, 0, 0),
    size=(0, 0.4, 0.4),
    source_time=td.GaussianPulse(freq0=freq0, fwidth=fwidth),
    polarization="Ex",
)

# create monitor
monitor = td.FieldMonitor(
    fields=["Ex", "Ey", "Ez"],
    center=(0, 0, 0),
    size=(td.inf, td.inf, 0),
    freqs=[freq0],
    name="field",
)

# Initialize simulation
sim = td.Simulation(
    size=sim_size,
    grid_spec=td.GridSpec.uniform(dl),
    structures=[square],
    sources=[source],
    monitors=[monitor],
    run_time=run_time,
    boundary_spec=td.BoundarySpec.all_sides(boundary=pml),
)


## Running simulation manually

For the most control, you can run the simulation through the Tidy3D web API.
Each simulation running on the server is identified by a `task_id`, which must be specified in the web API.
Let's walk through submitting a simulation this way.

In [3]:
# upload the simulation to our servers
task_id = web.upload(sim, task_name="webAPI", verbose=verbose)

# start the simulation running
web.start(task_id)

# monitor the simulation, dont move on to next line until completed.
web.monitor(task_id, verbose=verbose)

# download the results and load into a simulation data object for plotting, post processing etc.
sim_data = web.load(task_id, path="data/sim.hdf5", verbose=verbose)


13:32:00 PST Created task 'webAPI' with task_id                                 
             'fdve-45a9da50-f1fa-4aaf-ae4e-143da0402e66' and task_type 'FDTD'.

             View task using web UI at                                          
             ]8;id=735148;https://tidy3d.simulation.cloud/workbench?taskId=fdve-45a9da50-f1fa-4aaf-ae4e-143da0402e66\'https://tidy3d.simulation.cloud/workbench?]8;;\]8;id=992246;https://tidy3d.simulation.cloud/workbench?taskId=fdve-45a9da50-f1fa-4aaf-ae4e-143da0402e66\taskId]8;;\]8;id=735148;https://tidy3d.simulation.cloud/workbench?taskId=fdve-45a9da50-f1fa-4aaf-ae4e-143da0402e66\=]8;;\]8;id=828922;https://tidy3d.simulation.cloud/workbench?taskId=fdve-45a9da50-f1fa-4aaf-ae4e-143da0402e66\fdve]8;;\]8;id=735148;https://tidy3d.simulation.cloud/workbench?taskId=fdve-45a9da50-f1fa-4aaf-ae4e-143da0402e66\-45a9da50-f1f]8;;\
             ]8;id=735148;https://tidy3d.simulation.cloud/workbench?taskId=fdve-45a9da50-f1fa-4aaf-ae4e-143da0402e66\a-4aaf-ae4e-143da0402e66']8;;\.

Output()

13:32:01 PST status = queued

             To cancel the simulation, use 'web.abort(task_id)' or              
             'web.delete(task_id)' or abort/delete the task in the web UI.      
             Terminating the Python script will not stop the job running on the 
             cloud.

...

13:32:24 PST status = postprocess

13:32:28 PST status = success

13:32:29 PST View simulation result at                                          
             ]8;id=145753;https://tidy3d.simulation.cloud/workbench?taskId=fdve-45a9da50-f1fa-4aaf-ae4e-143da0402e66\'https://tidy3d.simulation.cloud/workbench?]8;;\]8;id=79985;https://tidy3d.simulation.cloud/workbench?taskId=fdve-45a9da50-f1fa-4aaf-ae4e-143da0402e66\taskId]8;;\]8;id=145753;https://tidy3d.simulation.cloud/workbench?taskId=fdve-45a9da50-f1fa-4aaf-ae4e-143da0402e66\=]8;;\]8;id=654552;https://tidy3d.simulation.cloud/workbench?taskId=fdve-45a9da50-f1fa-4aaf-ae4e-143da0402e66\fdve]8;;\]8;id=145753;https://tidy3d.simulation.cloud/workbench?taskId=fdve-45a9da50-f1fa-4aaf-ae4e-143da0402e66\-45a9da50-f1f]8;;\
             ]8;id=145753;https://tidy3d.simulation.cloud/workbench?taskId=fdve-45a9da50-f1fa-4aaf-ae4e-143da0402e66\a-4aaf-ae4e-143da0402e66']8;;\.

Output()

             loading simulation from data/sim.hdf5

While we broke down each of the individual steps above, one can also perform the entire process in one line by calling the [web.run()](https://docs.flexcompute.com/projects/tidy3d/en/latest/api/_autosummary/tidy3d.web.api.webapi.run.html) function as follows.

```python
sim_data = web.run(sim, task_name='webAPI', path='data/sim.hdf5')
```

(We won't run it again in this notebook to save time).

Sometimes this is more convenient, but other times it can be helpful to have the steps broken down one by one, for example if the simulation is long, you may want to [web.start](https://docs.flexcompute.com/projects/tidy3d/en/latest/api/_autosummary/tidy3d.web.api.webapi.start.html) and then exit the session and load the results at a later time using [web.load](https://docs.flexcompute.com/projects/tidy3d/en/latest/api/_autosummary/tidy3d.web.api.webapi.load.html).


## Job Container

The [web.Job](https://docs.flexcompute.com/projects/tidy3d/en/latest/api/_autosummary/tidy3d.web.api.container.Job.html) interface provides a more convenient way to manage single simuations, mainly because it eliminates the need for keeping track of the `task_id` and original [Simulation](https://docs.flexcompute.com/projects/tidy3d/en/latest/api/_autosummary/tidy3d.Simulation.html).

We can get the cost estimate of running the task before actually running it. This prevents us from accidentally running large jobs that we set up by mistake. The estimated cost is the maximum cost corresponding to running all the time steps.

In [4]:
# initializes job, puts task on server (but doesnt run it)
job = web.Job(simulation=sim, task_name="job", verbose=verbose)

# estimate the maximum cost
estimated_cost = web.estimate_cost(job.task_id)

13:32:30 PST Created task 'job' with task_id                                    
             'fdve-b4704762-5357-4bee-8e33-48ed1cf63197' and task_type 'FDTD'.

             View task using web UI at                                          
             ]8;id=86835;https://tidy3d.simulation.cloud/workbench?taskId=fdve-b4704762-5357-4bee-8e33-48ed1cf63197\'https://tidy3d.simulation.cloud/workbench?]8;;\]8;id=45216;https://tidy3d.simulation.cloud/workbench?taskId=fdve-b4704762-5357-4bee-8e33-48ed1cf63197\taskId]8;;\]8;id=86835;https://tidy3d.simulation.cloud/workbench?taskId=fdve-b4704762-5357-4bee-8e33-48ed1cf63197\=]8;;\]8;id=498657;https://tidy3d.simulation.cloud/workbench?taskId=fdve-b4704762-5357-4bee-8e33-48ed1cf63197\fdve]8;;\]8;id=86835;https://tidy3d.simulation.cloud/workbench?taskId=fdve-b4704762-5357-4bee-8e33-48ed1cf63197\-b4704762-535]8;;\
             ]8;id=86835;https://tidy3d.simulation.cloud/workbench?taskId=fdve-b4704762-5357-4bee-8e33-48ed1cf63197\7-4bee-8e33-48ed1cf63197']8;;\.

Output()

13:32:31 PST Maximum FlexCredit cost: 0.025. Minimum cost depends on task       
             execution details. Use 'web.real_cost(task_id)' to get the billed  
             FlexCredit cost after a simulation run.

While [Job](https://docs.flexcompute.com/projects/tidy3d/en/latest/api/_autosummary/tidy3d.web.api.container.Job.html) has methods with names identical to the web API functions above, which give some more granular control, it is often most convenient to call `.run()` so we will do that now.

In [5]:
# start job, monitor, and load results
sim_data = job.run(path="data/sim.hdf5")

             status = success

Output()

13:32:32 PST loading simulation from data/sim.hdf5

Another convenient thing about [Job](https://docs.flexcompute.com/projects/tidy3d/en/latest/api/_autosummary/tidy3d.web.api.container.Job.html) objects is that they can be saved and loaded just like other Tidy3d components.

This is convenient if you want to save and load up the results of a job that has already finished, without needing to know the `task_id`.

In [6]:
# saves the job metadata to a single file
job.to_file("data/job.json")

# can exit session, break here, or continue in new session.

# load the job metadata from file
job_loaded = web.Job.from_file("data/job.json")

# download the data from the server and load it into a SimulationData object.
sim_data = job_loaded.load(path="data/sim.hdf5")


Output()

             loading simulation from data/sim.hdf5

## Batch Processing

Commonly one needs to submit a batch of Simulations.
One way to approach this using the web API is to upload, start, monitor, and load a series of tasks one by one, but this is clumsy and you can lose your data if the session gets interrupted.

A better way is to use the built-in [Batch](https://docs.flexcompute.com/projects/tidy3d/en/latest/api/_autosummary/tidy3d.web.api.container.Batch.html) object.
The batch object is like a [Job](https://docs.flexcompute.com/projects/tidy3d/en/latest/api/_autosummary/tidy3d.web.api.container.Job.html), but stores task metadata for a series of simulations.

In [7]:
# make a dictionary of  {task name : simulation} for demonstration
sims = {f"sim_{i}": sim for i in range(3)}

# initialize a batch and run them all
batch = web.Batch(simulations=sims, verbose=verbose)

# run the batch and store all of the data in the `data/` dir.
batch_results = batch.run(path_dir="data")


             Created task 'sim_0' with task_id                                  
             'fdve-a9f944f4-d933-4972-88b6-616217c6b7cf' and task_type 'FDTD'.

             View task using web UI at                                          
             ]8;id=844698;https://tidy3d.simulation.cloud/workbench?taskId=fdve-a9f944f4-d933-4972-88b6-616217c6b7cf\'https://tidy3d.simulation.cloud/workbench?]8;;\]8;id=879234;https://tidy3d.simulation.cloud/workbench?taskId=fdve-a9f944f4-d933-4972-88b6-616217c6b7cf\taskId]8;;\]8;id=844698;https://tidy3d.simulation.cloud/workbench?taskId=fdve-a9f944f4-d933-4972-88b6-616217c6b7cf\=]8;;\]8;id=276298;https://tidy3d.simulation.cloud/workbench?taskId=fdve-a9f944f4-d933-4972-88b6-616217c6b7cf\fdve]8;;\]8;id=844698;https://tidy3d.simulation.cloud/workbench?taskId=fdve-a9f944f4-d933-4972-88b6-616217c6b7cf\-a9f944f4-d93]8;;\
             ]8;id=844698;https://tidy3d.simulation.cloud/workbench?taskId=fdve-a9f944f4-d933-4972-88b6-616217c6b7cf\3-4972-88b6-616217c6b7cf']8;;\.

Output()

13:32:33 PST Created task 'sim_1' with task_id                                  
             'fdve-adde2c75-7124-440c-a92c-c9c619bc3d83' and task_type 'FDTD'.

             View task using web UI at                                          
             ]8;id=966485;https://tidy3d.simulation.cloud/workbench?taskId=fdve-adde2c75-7124-440c-a92c-c9c619bc3d83\'https://tidy3d.simulation.cloud/workbench?]8;;\]8;id=267364;https://tidy3d.simulation.cloud/workbench?taskId=fdve-adde2c75-7124-440c-a92c-c9c619bc3d83\taskId]8;;\]8;id=966485;https://tidy3d.simulation.cloud/workbench?taskId=fdve-adde2c75-7124-440c-a92c-c9c619bc3d83\=]8;;\]8;id=991729;https://tidy3d.simulation.cloud/workbench?taskId=fdve-adde2c75-7124-440c-a92c-c9c619bc3d83\fdve]8;;\]8;id=966485;https://tidy3d.simulation.cloud/workbench?taskId=fdve-adde2c75-7124-440c-a92c-c9c619bc3d83\-adde2c75-712]8;;\
             ]8;id=966485;https://tidy3d.simulation.cloud/workbench?taskId=fdve-adde2c75-7124-440c-a92c-c9c619bc3d83\4-440c-a92c-c9c619bc3d83']8;;\.

...

13:32:34 PST Started working on Batch.

13:32:35 PST Maximum FlexCredit cost: 0.075 for the whole batch.

             Use 'Batch.real_cost()' to get the billed FlexCredit cost after the
             Batch has completed.

Output()

             Batch complete.

When the batch is completed, the output is not a [SimulationData](https://docs.flexcompute.com/projects/tidy3d/en/latest/api/_autosummary/tidy3d.SimulationData.html) but rather a [BatchData](https://docs.flexcompute.com/projects/tidy3d/en/latest/api/_autosummary/tidy3d.web.api.container.BatchData.html).  The data within this [BatchData](https://docs.flexcompute.com/projects/tidy3d/en/latest/api/_autosummary/tidy3d.web.api.container.BatchData.html) object can either be indexed directly `batch_results[task_name]` or can be looped through `batch_results.items()` to get the [SimulationData](https://docs.flexcompute.com/projects/tidy3d/en/latest/api/_autosummary/tidy3d.SimulationData.html) for each task.

This was chosen to reduce the memory strain from loading all [SimulationData](https://docs.flexcompute.com/projects/tidy3d/en/latest/api/_autosummary/tidy3d.SimulationData.html) objects at once.

Alternatively, the batch can be looped through (several times) using the `.items()` method, similar to a dictionary.

In [8]:
# grab the sum of intensities in the simulation one by one (to save memory)
intensities = {}
for task_name, sim_data in batch_results.items():
    intensity = sim_data.get_intensity("field").sel(f=freq0)
    sum_intensity = float(intensity.sum(("x", "y")).values)
    intensities[task_name] = sum_intensity

print(intensities)


Output()

13:32:36 PST loading simulation from                                            
             data/fdve-a9f944f4-d933-4972-88b6-616217c6b7cf.hdf5

/tmp/ipykernel_745201/711851585.py:5: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  sum_intensity = float(intensity.sum(("x", "y")).values)


Output()

13:32:37 PST loading simulation from                                            
             data/fdve-adde2c75-7124-440c-a92c-c9c619bc3d83.hdf5

/tmp/ipykernel_745201/711851585.py:5: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  sum_intensity = float(intensity.sum(("x", "y")).values)


Output()

             loading simulation from                                            
             data/fdve-f0b99b06-e500-4fe0-8153-fdd9952e1e48.hdf5

{'sim_0': 6377911.0, 'sim_1': 6377911.0, 'sim_2': 6377911.0}


/tmp/ipykernel_745201/711851585.py:5: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  sum_intensity = float(intensity.sum(("x", "y")).values)


## Asynchronous Batching

Finally, one can make use of the [asyncio package](https://realpython.com/async-io-python/) to perform asynchronous processing of several simulations.

For this purpose, a [web.run_async](https://docs.flexcompute.com/projects/tidy3d/en/latest/api/_autosummary/tidy3d.web.api.asynchronous.run_async.html) function is provided, which works like the regular `web.run` but accepts a dictionary of simulations. 

Here is the previous example repeated using this feature.

In [9]:
batch_results = web.run_async(simulations=sims, verbose=verbose)


             Created task 'sim_0' with task_id                                  
             'fdve-14d8eb58-74a0-468b-83bb-13962c421123' and task_type 'FDTD'.

             View task using web UI at                                          
             ]8;id=595331;https://tidy3d.simulation.cloud/workbench?taskId=fdve-14d8eb58-74a0-468b-83bb-13962c421123\'https://tidy3d.simulation.cloud/workbench?]8;;\]8;id=435155;https://tidy3d.simulation.cloud/workbench?taskId=fdve-14d8eb58-74a0-468b-83bb-13962c421123\taskId]8;;\]8;id=595331;https://tidy3d.simulation.cloud/workbench?taskId=fdve-14d8eb58-74a0-468b-83bb-13962c421123\=]8;;\]8;id=479849;https://tidy3d.simulation.cloud/workbench?taskId=fdve-14d8eb58-74a0-468b-83bb-13962c421123\fdve]8;;\]8;id=595331;https://tidy3d.simulation.cloud/workbench?taskId=fdve-14d8eb58-74a0-468b-83bb-13962c421123\-14d8eb58-74a]8;;\
             ]8;id=595331;https://tidy3d.simulation.cloud/workbench?taskId=fdve-14d8eb58-74a0-468b-83bb-13962c421123\0-468b-83bb-13962c421123']8;;\.

Output()

13:32:38 PST Created task 'sim_1' with task_id                                  
             'fdve-39450e0f-bf4f-402c-becd-55ad27a05f3b' and task_type 'FDTD'.

             View task using web UI at                                          
             ]8;id=21616;https://tidy3d.simulation.cloud/workbench?taskId=fdve-39450e0f-bf4f-402c-becd-55ad27a05f3b\'https://tidy3d.simulation.cloud/workbench?]8;;\]8;id=292258;https://tidy3d.simulation.cloud/workbench?taskId=fdve-39450e0f-bf4f-402c-becd-55ad27a05f3b\taskId]8;;\]8;id=21616;https://tidy3d.simulation.cloud/workbench?taskId=fdve-39450e0f-bf4f-402c-becd-55ad27a05f3b\=]8;;\]8;id=27173;https://tidy3d.simulation.cloud/workbench?taskId=fdve-39450e0f-bf4f-402c-becd-55ad27a05f3b\fdve]8;;\]8;id=21616;https://tidy3d.simulation.cloud/workbench?taskId=fdve-39450e0f-bf4f-402c-becd-55ad27a05f3b\-39450e0f-bf4]8;;\
             ]8;id=21616;https://tidy3d.simulation.cloud/workbench?taskId=fdve-39450e0f-bf4f-402c-becd-55ad27a05f3b\f-402c-becd-55ad27a05f3b']8;;\.

...

13:32:39 PST Started working on Batch.

13:32:40 PST Maximum FlexCredit cost: 0.075 for the whole batch.

             Use 'Batch.real_cost()' to get the billed FlexCredit cost after the
             Batch has completed.

Output()

13:32:41 PST Batch complete.

In [10]:
# grab the sum of intensities in the simulation one by one (to save memory)
intensities = {}
for task_name, sim_data in batch_results.items():
    intensity = sim_data.get_intensity("field").sel(f=freq0)
    sum_intensity = float(intensity.sum(("x", "y")).values)
    intensities[task_name] = sum_intensity

print(intensities)


Output()

             loading simulation from                                            
             ./fdve-14d8eb58-74a0-468b-83bb-13962c421123.hdf5

/tmp/ipykernel_745201/711851585.py:5: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  sum_intensity = float(intensity.sum(("x", "y")).values)


Output()

13:32:42 PST loading simulation from                                            
             ./fdve-39450e0f-bf4f-402c-becd-55ad27a05f3b.hdf5

/tmp/ipykernel_745201/711851585.py:5: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  sum_intensity = float(intensity.sum(("x", "y")).values)


Output()

             loading simulation from                                            
             ./fdve-d591056e-59e4-48eb-9183-3e7066e753a3.hdf5

{'sim_0': 6377911.0, 'sim_1': 6377911.0, 'sim_2': 6377911.0}


/tmp/ipykernel_745201/711851585.py:5: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  sum_intensity = float(intensity.sum(("x", "y")).values)


After going through this tutorial, you have learned how to run simulations with Tidy3D via the web API. If you are new to Tidy3D or the finite-difference time-domain (FDTD) method, we highly recommend going through our [FDTD101](https://www.flexcompute.com/fdtd101/) tutorials and [example library](https://www.flexcompute.com/tidy3d/examples/) before starting your own simulation adventure. 